
# 01 · Teste de acesso às fontes
Verifica se o Databricks Free Edition acessa as URLs das fontes mapeadas.

## 1. Teste de acesso

`stream=True` lê apenas o início da resposta: código HTTP e tipo de conteúdo.

In [0]:
import requests
from datetime import datetime

# (fonte, descrição, URL)
fontes = [
    ("SEFAZ-RS", "Portal Receita Dados",
     "https://receitadados.sefaz.rs.gov.br/"),
    ("Receita Federal", "Diretório dados abertos CNPJ",
     "https://arquivos.receitafederal.gov.br/dados/cnpj/dados_abertos_cnpj/"),
    ("IBGE", "API Localidades - municípios RS",
     "https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios"),
    ("IBGE", "API SIDRA - PIB municipal (tabela 5938)",
     "https://apisidra.ibge.gov.br/values/t/5938/n6/4314902/v/37/p/last"),
    ("Comex Stat", "Exportações por município 2024",
     "https://balanca.economia.gov.br/balanca/bd/comexstat-bd/mun/EXP_2024_MUN.csv"),
    ("MTE/PDET", "Portal RAIS (plano B)",
     "https://pdet.mte.gov.br/"),
]

resultados = []
for fonte, descricao, url in fontes:
    try:
        r = requests.get(url, timeout=20, stream=True,
                         headers={"User-Agent": "Mozilla/5.0"})
        amostra = next(r.iter_content(200), b"")
        resultados.append((fonte, descricao, url, r.status_code,
                           "OK" if r.ok else "HTTP erro",
                           r.headers.get("Content-Type", ""),
                           amostra[:80].decode("utf-8", "ignore")))
        r.close()
    except Exception as e:
        resultados.append((fonte, descricao, url, None, "BLOQUEADO/FALHA",
                           "", f"{type(e).__name__}: {str(e)[:120]}"))

display(spark.createDataFrame(resultados,
    "fonte string, descricao string, url string, status_http int, "
    "resultado string, content_type string, detalhe string"))
print(f"Teste executado em {datetime.now():%d/%m/%Y %H:%M}")

## 2. Reteste sem validação de certificado

Portais de governo usam certificado ICP-Brasil, não reconhecido pelo ambiente. `verify=False`
distingue fonte indisponível de certificado não validado. Aplicado apenas a dados públicos, sem
credencial.

In [0]:
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

retestes = [
    ("IBGE", "API Agregados v3 - PIB municipal RS (tabela 5938)",
     "https://servicodados.ibge.gov.br/api/v3/agregados/5938/periodos/-1/variaveis/37?localidades=N6[N3[43]]", True),
    ("Comex Stat", "Exportações por município 2024 (sem verificação SSL)",
     "https://balanca.economia.gov.br/balanca/bd/comexstat-bd/mun/EXP_2024_MUN.csv", False),
    ("MTE/PDET", "Portal RAIS (sem verificação SSL)",
     "https://pdet.mte.gov.br/", False),
    ("Receita Federal", "Diretório CNPJ (nova tentativa)",
     "https://arquivos.receitafederal.gov.br/dados/cnpj/dados_abertos_cnpj/", False),
]

resultados2 = []
for fonte, descricao, url, verificar in retestes:
    try:
        r = requests.get(url, timeout=30, stream=True, verify=verificar,
                         headers={"User-Agent": "Mozilla/5.0"})
        amostra = next(r.iter_content(200), b"")
        resultados2.append((fonte, descricao, r.status_code,
                            "OK" if r.ok else "HTTP erro",
                            r.headers.get("Content-Length", ""),
                            amostra[:100].decode("utf-8", "ignore")))
        r.close()
    except Exception as e:
        resultados2.append((fonte, descricao, None, "BLOQUEADO/FALHA", "",
                            f"{type(e).__name__}: {str(e)[:150]}"))

display(spark.createDataFrame(resultados2,
    "fonte string, descricao string, status_http int, resultado string, "
    "tamanho_bytes string, detalhe string"))

## 3. Escrita no volume

Gravar no volume. 

In [0]:
import json

url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios"
destino = "/Volumes/mvp_pipeline_vf/bronze/arquivos_brutos/teste/ibge_municipios_rs.json"

dados = requests.get(url, timeout=30).json()
dbutils.fs.mkdirs("/Volumes/mvp_pipeline_vf/bronze/arquivos_brutos/teste")
with open(destino, "w", encoding="utf-8") as f:
    json.dump(dados, f, ensure_ascii=False)

print(f"Municípios gravados: {len(dados)}")
display(dbutils.fs.ls("/Volumes/mvp_pipeline_vf/bronze/arquivos_brutos/teste"))

## 4. Fontes adotadas

O escopo final do MVP usa quatro origens, todas com acesso confirmado:

| Fonte | Conteúdo | Formato |
| --- | --- | --- |
| SEFAZ-RS / Receita Dados | ICMS por CNAE, arrecadação por município, desonerações, cadastro de contribuintes | CSV |
| IBGE | Municípios do RS | API REST |
| DEE-RS | PIB e PIB per capita municipal | CSV |
| Sebrae RS | Classificação de CNAEs por cadeia produtiva | XLSX |

